# Lab 1 — Neural Network trên Google Colab
Chọn **Runtime → Change runtime type → T4 GPU**, điền MSSV và chạy **Run all**. Không dùng eval để tuning.

In [4]:
# Nếu mở notebook bằng link GitHub, Colab cần clone repo trước.
REPO_URL = "https://github.com/TrKhuyn/K4-Track4-Day1-Neuralnetwork.git"
MSSV = "2A202602682"

assert REPO_URL.startswith(("http://", "https://")), \
    "REPO_URL phải là URL GitHub đầy đủ"
assert MSSV.strip() and not MSSV.startswith("YOUR_"), \
    "Hãy điền MSSV thật"
import os,sys,subprocess,shutil,json,math
from pathlib import Path
if not Path('/content/lab1').exists(): subprocess.run(['git','clone',REPO_URL,'/content/lab1'],check=True)
os.chdir('/content/lab1/code'); REPO_ROOT=Path('/content/lab1')
from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive')/f'submission_{MSSV}'
for d in ('code','figures','results'): (OUT/d).mkdir(parents=True,exist_ok=True)
for p in Path('.').glob('*.py'): shutil.copy2(p,OUT/'code'/p.name)
import numpy as np, torch
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.__version__,device,torch.cuda.get_device_name(0) if device.type=='cuda' else '')
from data import prepare_data
from model import MLP,EXPECTED_PARAMS,count_params,activation_stats
from train import DEFAULT_CFG,evaluate,run_experiment,final_eval,compute_loss
from plots import plot_run,plot_compare
from results_table import save_result,to_row,write_xlsx


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.11.0+cu130 cuda Tesla T4


## Part 0–1 — Dữ liệu và sanity checks
Validation được tách phân tầng từ train; standardizer chỉ fit trên training subset.

In [5]:
processed=REPO_ROOT/'data'/'processed'
if not (processed/'train.npz').exists(): subprocess.run([sys.executable,'scripts/split_data.py'],cwd=REPO_ROOT,check=True)
data=prepare_data(str(device),0.2,42,str(processed))
model=MLP(init='he').to(device); assert count_params(model)==EXPECTED_PARAMS[(256,128)]
assert model(torch.randn(8,54,device=device)).shape==(8,7)
print('step0 / ln7:',evaluate(model,data['X_val'],data['y_val'])['loss'],math.log(7))
print('activation std:',activation_stats(model,data['X_val'][:512]))
tiny=MLP(init='he').to(device); opt=torch.optim.Adam(tiny.parameters(),lr=1e-2); losses=[]
for _ in range(400):
    opt.zero_grad(set_to_none=True); loss=compute_loss(tiny(data['X_tr'][:20]),data['y_tr'][:20],'ce'); loss.backward(); opt.step(); losses.append(loss.item())
print('overfit 20:',losses[0],losses[-1]); assert losses[-1]<.05


train=(371847, 54), val=(92962, 54), eval=(116203, 54); majority-val-acc=0.4876
step0 / ln7: 2.014487890129166 1.9459101490553132
activation std: [0.404516339302063, 0.4175407290458679]
overfit 20: 2.1190881729125977 1.6331636061295285e-06


## Part 2–3 — Baseline, seed noise và thí nghiệm
Mỗi thí nghiệm đổi một yếu tố, cùng split và 20 epoch. Hãy ghi dự đoán trước và nhận xét sau vào báo cáo.

In [6]:
def run_save(cfg):
    r=run_experiment(cfg,data); save_result(r,str(OUT/'results')); plot_run(r,str(OUT/'figures'/f"{cfg['exp_id']}.png")); return r
base={**DEFAULT_CFG,'lr':.05,'epochs':20}
bases=[run_save({**base,'seed':s,'exp_id':f'base-s{s}'}) for s in (1,2,3)]
f=np.array([r['summary']['val_macro_f1'] for r in bases]); print('baseline mean/std/2sigma:',f.mean(),f.std(ddof=1),2*f.std(ddof=1))
cfgs=[
 {**base,'exp_id':'opt-adam','group':'optimizer','description':'Adam','optimizer':'adam','lr':1e-3},
 {**base,'exp_id':'opt-adamw','group':'optimizer','description':'AdamW','optimizer':'adamw','lr':1e-3,'weight_decay':1e-4},
 {**base,'exp_id':'dropout-02','group':'dropout','description':'Dropout .2','dropout':.2},
 {**base,'exp_id':'clip-10','group':'clipping','description':'Clip norm 1','clip_norm':1.},
 {**base,'exp_id':'loss-mse','group':'loss','description':'MSE one-hot','loss':'mse'},
 {**base,'exp_id':'init-xavier','group':'init','description':'Xavier','init':'xavier'}]
others=[run_save(c) for c in cfgs]; results=bases+others
for group in ('optimizer','dropout','clipping','loss','init'):
    rs=[bases[0]]+[r for r in others if r['cfg']['group']==group]; plot_compare(rs,'val_macro_f1',str(OUT/'figures'/f'compare_{group}.png'),group)
[(r['cfg']['exp_id'],round(r['summary']['val_macro_f1'],4),r['summary']['best_epoch']) for r in results]


[base-s1] 01/20 train=0.4923 val=0.4949 f1=0.6040
[base-s1] 02/20 train=0.4272 val=0.4321 f1=0.6943
[base-s1] 03/20 train=0.4218 val=0.4294 f1=0.6782
[base-s1] 04/20 train=0.3736 val=0.3815 f1=0.7314
[base-s1] 05/20 train=0.3686 val=0.3773 f1=0.7333
[base-s1] 06/20 train=0.3369 val=0.3455 f1=0.7572
[base-s1] 07/20 train=0.3284 val=0.3398 f1=0.7678
[base-s1] 08/20 train=0.3198 val=0.3311 f1=0.7923
[base-s1] 09/20 train=0.2836 val=0.2973 f1=0.7945
[base-s1] 10/20 train=0.3001 val=0.3146 f1=0.7849
[base-s1] 11/20 train=0.2757 val=0.2887 f1=0.8058
[base-s1] 12/20 train=0.2961 val=0.3108 f1=0.7817
[base-s1] 13/20 train=0.2744 val=0.2901 f1=0.8220
[base-s1] 14/20 train=0.2705 val=0.2853 f1=0.8159
[base-s1] 15/20 train=0.2515 val=0.2675 f1=0.8186
[base-s1] 16/20 train=0.2563 val=0.2739 f1=0.8273
[base-s1] 17/20 train=0.2386 val=0.2570 f1=0.8358
[base-s1] 18/20 train=0.3034 val=0.3222 f1=0.8192
[base-s1] 19/20 train=0.2356 val=0.2554 f1=0.8449
[base-s1] 20/20 train=0.2380 val=0.2576 f1=0.8358


[('base-s1', 0.8449, 19),
 ('base-s2', 0.8385, 20),
 ('base-s3', 0.8463, 19),
 ('opt-adam', 0.8437, 18),
 ('opt-adamw', 0.8444, 18),
 ('dropout-02', 0.7955, 20),
 ('clip-10', 0.8407, 19),
 ('loss-mse', 0.6834, 20),
 ('init-xavier', 0.8409, 20)]

## Part 4 — Chọn bằng validation và đánh giá eval
Cấu hình cuối được chọn trước khi đọc điểm eval.

In [7]:
winner=max(results,key=lambda r:r['summary']['val_macro_f1']); print('selected by val:',winner['cfg']['exp_id'])
pred=OUT/'predictions_eval.csv'; final_eval(winner['cfg'],winner,data,str(pred))
ev=OUT/'eval_result.json'; subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(pred),'--out',str(ev)],cwd=REPO_ROOT,check=True)
scores=json.loads(ev.read_text()); print('final eval:',scores['accuracy'],scores['macro_f1'])
bp=OUT/'predictions_baseline_eval.csv'; final_eval(bases[0]['cfg'],bases[0],data,str(bp))
be=OUT/'eval_result_baseline.json'; subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(bp),'--out',str(be)],cwd=REPO_ROOT,check=True)
bs=json.loads(be.read_text()); print('baseline eval:',bs['accuracy'],bs['macro_f1'])
rows=[to_row(r,scores if r is winner else (bs if r is bases[0] else None)) for r in results]
write_xlsx(rows,str(REPO_ROOT/'templates'/'experiment_table_template.xlsx'),str(OUT/'experiments.xlsx'))
shutil.copy2(REPO_ROOT/'templates'/'REPORT_TEMPLATE.md',OUT/'REPORT.md'); shutil.copy2(REPO_ROOT/'code'/'lab.ipynb',OUT/'code'/'lab.ipynb')
print('Output:',OUT); print('Viết REPORT.md và nhận xét dựa trên số liệu thật trước khi nộp.')


selected by val: base-s3
final eval: 0.9027391719663004 0.8526773031573471
baseline eval: 0.8963796115418707 0.8484250320507591
Output: /content/drive/MyDrive/submission_2A202602682
Viết REPORT.md và nhận xét dựa trên số liệu thật trước khi nộp.
